# 06 — Baseline eval: teacher và student base chưa distill

Chấm điểm mọi model gốc trên đúng bộ benchmark test mà `00_main_results` dùng, bằng chính
`src/evaluation`, nên số ở đây so trực tiếp được với bảng main results.

- Không train: mỗi model chỉ là một lượt forward + probe logistic cho nhóm classification.
- Pooling theo từng model (`last_token` cho Qwen3-Embedding, `cls` cho encoder), giống lúc distill.
- Mỗi (model, seed) ghi `results.json` riêng nên chạy lại là resume, không tính lại phần đã xong.
- `SEEDS` giữ đúng format của `00_main_results` (mean ± sample std, cột `_n`). Lưu ý: eval baseline
  không train gì cả, embedding và probe lbfgs đều tất định, nên nhiều seed cho ra cùng một số và
  cột std về 0.00 — nó là bằng chứng "không có phương sai", không phải phép đo phương sai. Đặt
  `SEEDS = [42]` nếu không muốn trả giá 3x compute cho kết luận đó.
- `avg_all` chỉ trung bình 9 benchmark sentence-level; 5 retrieval benchmark chỉ đi vào `avg_retrieval`.


In [1]:
# 1. Cấu hình
from datetime import datetime
from pathlib import Path
from zoneinfo import ZoneInfo

REPO_URL = "https://github.com/duncan-nguyen/embedding-kd.git"

# Teacher và student base của ba cặp trong 00_main_results, chưa qua distill.
BASELINES = {
    "Qwen/Qwen3-Embedding-0.6B": {"pooling": "last_token", "dtype": "bfloat16"},
    "Qwen/Qwen3-Embedding-4B": {"pooling": "last_token", "dtype": "bfloat16"},
    "BAAI/bge-m3": {"pooling": "cls", "dtype": "bfloat16"},
    "nreimers/MiniLMv2-L6-H384-distilled-from-BERT-Base": {"pooling": "cls", "dtype": "float32"},
    "nreimers/MiniLMv2-L6-H768-distilled-from-BERT-Base": {"pooling": "cls", "dtype": "float32"},
    "google-bert/bert-base-uncased": {"pooling": "cls", "dtype": "float32"},
}
# Cắt bớt list này để chỉ chạy một phần.
MODELS = list(BASELINES)

SEEDS = [42, 43, 44]
EVAL_RETRIEVAL = False
CUDA_VISIBLE_DEVICES = "0"
AUTO_FETCH_DATA = True
STOP_ON_ERROR = True

RUN_STAMP = datetime.now(ZoneInfo("Asia/Ho_Chi_Minh")).strftime("%Y%m%d-%H%M%S")
# Điền tên run cũ để resume sau khi restart runtime; None tạo run mới.
RUN_NAME_OVERRIDE = None
RUN_NAME = RUN_NAME_OVERRIDE or f"baselines_{len(SEEDS)}seeds_{RUN_STAMP}"

assert MODELS and set(MODELS) <= set(BASELINES), f"MODELS phải nằm trong BASELINES: {MODELS}"
assert SEEDS and len(SEEDS) == len(set(SEEDS)), f"SEEDS phải không rỗng và không trùng: {SEEDS}"
print(f"Run: {RUN_NAME}")
print(f"Seeds: {SEEDS}")
print(f"Models ({len(MODELS)}):")
for name in MODELS:
    print(f"  {name}  pooling={BASELINES[name]['pooling']}  dtype={BASELINES[name]['dtype']}")


Run: baselines_3seeds_20260904-122835
Seeds: [42, 43, 44]
Models (6):
  Qwen/Qwen3-Embedding-0.6B  pooling=last_token  dtype=bfloat16
  Qwen/Qwen3-Embedding-4B  pooling=last_token  dtype=bfloat16
  BAAI/bge-m3  pooling=cls  dtype=bfloat16
  nreimers/MiniLMv2-L6-H384-distilled-from-BERT-Base  pooling=cls  dtype=float32
  nreimers/MiniLMv2-L6-H768-distilled-from-BERT-Base  pooling=cls  dtype=float32
  google-bert/bert-base-uncased  pooling=cls  dtype=float32


In [ ]:
# 2. Dùng repo hiện tại hoặc clone trên Colab; cài dependencies.
import subprocess
import sys

cwd = Path.cwd().resolve()
if (cwd / "main.py").is_file() and (cwd / "distiller.py").is_file():
    PROJECT_DIR = cwd
else:
    clone_parent = Path("/content") if Path("/content").is_dir() else cwd
    PROJECT_DIR = clone_parent / "embedding-kd"
    if PROJECT_DIR.exists():
        assert (PROJECT_DIR / "main.py").is_file(), f"Repo không hợp lệ: {PROJECT_DIR}"
    else:
        subprocess.run(["git", "clone", REPO_URL, str(PROJECT_DIR)], check=True)

head_before = subprocess.run(
    ["git", "-C", str(PROJECT_DIR), "rev-parse", "--short", "HEAD"],
    check=True, capture_output=True, text=True,
).stdout.strip()
subprocess.run(["git", "-C", str(PROJECT_DIR), "pull", "--ff-only"], check=True)
head_after = subprocess.run(
    ["git", "-C", str(PROJECT_DIR), "rev-parse", "--short", "HEAD"],
    check=True, capture_output=True, text=True,
).stdout.strip()
print(f"Git HEAD: {head_before} -> {head_after}")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-r", str(PROJECT_DIR / "requirements.txt")],
    check=True,
)


In [ ]:
# 3. Output, GPU và dữ liệu
import os

os.environ["CUDA_VISIBLE_DEVICES"] = CUDA_VISIBLE_DEVICES
os.environ["TOKENIZERS_PARALLELISM"] = "false"
import torch

RUN_ROOT = PROJECT_DIR / "runs" / RUN_NAME
RETRIEVAL_DIR = PROJECT_DIR / "data" / "test_set" / "retrieval"

for split in ("train_set", "test_set"):
    split_dir = PROJECT_DIR / "data" / split
    assert split_dir.is_dir() and any(split_dir.glob("*.csv")), f"Thiếu data: {split_dir}"

missing_retrieval = [
    name for name in ("arguana", "fiqa", "scidocs", "scifact", "nfcorpus")
    if not (RETRIEVAL_DIR / name / "corpus.csv").is_file()
]
if EVAL_RETRIEVAL and missing_retrieval:
    if not AUTO_FETCH_DATA:
        raise FileNotFoundError(f"Thiếu retrieval data: {missing_retrieval}")
    subprocess.run(
        [sys.executable, "scripts/data/download_retrieval_benchmarks.py"],
        cwd=PROJECT_DIR, check=True,
    )

if not torch.cuda.is_available():
    raise RuntimeError("Hãy bật GPU runtime trước khi chạy.")
DEVICE = torch.device("cuda:0")
props = torch.cuda.get_device_properties(0)
print(f"cuda:0: {props.name} ({props.total_memory / 2**30:.1f} GiB)")

RUN_ROOT.mkdir(parents=True, exist_ok=True)
print(f"Output root: {RUN_ROOT}")


In [ ]:
# 4. Eval từng model: một lượt test split, ghi results.json để resume.
import gc
import json
import random
import time

import numpy as np

if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

from transformers import AutoModel, AutoTokenizer
from transformers import __version__ as transformers_version

from distiller import KnowledgeDistiller
from src.evaluation.evaluation_automodel import (
    eval_classification_task,
    eval_pair_task,
    eval_sts_task,
    test_cls_tasks,
    test_pair_tasks,
    test_sts_tasks,
)
from src.evaluation.retrieval import eval_retrieval_task, test_retrieval_tasks
from src.pooling import pool_sentence_embedding

# transformers >= 5 đổi tên keyword nạp dtype.
DTYPE_ARG = "dtype" if int(transformers_version.split(".")[0]) >= 5 else "torch_dtype"
DTYPES = {"float32": torch.float32, "float16": torch.float16, "bfloat16": torch.bfloat16}


class PooledEncoder(torch.nn.Module):
    """AutoModel + pooling của chính model đó.

    `_embed_texts` đọc khóa "pooled" nếu forward trả dict, nên teacher decoder được
    chấm ở last token thay vì bị ép về vị trí CLS như student encoder.
    """

    def __init__(self, name, pooling, dtype):
        super().__init__()
        self.backbone = AutoModel.from_pretrained(
            name, trust_remote_code=True, **{DTYPE_ARG: DTYPES[dtype]}
        )
        self.pooling = pooling

    @property
    def device(self):
        return next(self.backbone.parameters()).device

    def forward(self, input_ids, attention_mask):
        output = self.backbone(input_ids=input_ids, attention_mask=attention_mask)
        pooled = pool_sentence_embedding(
            output.last_hidden_state, attention_mask, self.pooling
        )
        return {"pooled": pooled}


def score_from_payload(family, raw_values):
    if family == "classification":
        return float(raw_values["f1"])
    if family == "pair":
        return float(raw_values["average_precision"])
    if family == "sts":
        return float(raw_values)
    if family == "retrieval":
        return float(raw_values["ndcg_at_10"])
    raise KeyError(f"Unknown family: {family}")


def evaluate_baseline(name, settings, seed):
    # Không có bước train nào để seed, nhưng cứ ghim mọi nguồn ngẫu nhiên của
    # process lại thì hai seed ra khác nhau mới là chuyện đáng điều tra.
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    encoder = PooledEncoder(name, settings["pooling"], settings["dtype"]).to(DEVICE).eval()
    tokenizer = AutoTokenizer.from_pretrained(name, trust_remote_code=True, use_fast=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    try:
        # Threshold của nhóm pair quét ngay trên test (không có run validation nào để
        # kế thừa), đúng mặc định pair_threshold_source="test" của main.py.
        pair, _ = eval_pair_task(encoder, test_pair_tasks, tokenizer)
        results = {
            "classification": eval_classification_task(encoder, test_cls_tasks, tokenizer),
            "pair": pair,
            "sts": eval_sts_task(encoder, test_sts_tasks, tokenizer),
            "retrieval": (
                eval_retrieval_task(encoder, test_retrieval_tasks, tokenizer)
                if EVAL_RETRIEVAL else {}
            ),
            "pair_threshold_source": "test",
        }
    finally:
        del encoder
        gc.collect()
        torch.cuda.empty_cache()

    scores = {
        KnowledgeDistiller._benchmark_name(path, "test"): score_from_payload(family, values)
        for family in ("classification", "pair", "sts", "retrieval")
        for path, values in results[family].items()
    }
    averages = KnowledgeDistiller._benchmark_group_averages(scores)
    results["summary"] = {key: group["score"] for key, group in averages.items()}
    return results


def result_path_of(name, seed):
    return RUN_ROOT / name.replace("/", "__") / f"seed_{seed}" / "results.json"


JOBS = [(name, seed) for name in MODELS for seed in SEEDS]
status = []
for position, (name, seed) in enumerate(JOBS, start=1):
    result_path = result_path_of(name, seed)
    if result_path.is_file():
        print(f"[SKIP] {name} seed={seed} đã có results.json")
        status.append({"model": name, "seed": seed, "status": "skipped_complete", "seconds": 0.0})
        continue
    print("\n" + "#" * 88)
    print(f"JOB {position}/{len(JOBS)}: {name} — seed {seed}")
    print("#" * 88)
    started = time.perf_counter()
    try:
        results = evaluate_baseline(name, BASELINES[name], seed)
    except Exception as error:
        print(f"[FAILED] {name} seed={seed}: {error}")
        status.append({"model": name, "seed": seed, "status": "failed", "seconds": time.perf_counter() - started})
        if STOP_ON_ERROR:
            raise
        continue
    elapsed = time.perf_counter() - started
    result_path.parent.mkdir(parents=True, exist_ok=True)
    result_path.write_text(
        json.dumps({"model": name, "seed": seed, **BASELINES[name], "test": results}, indent=2),
        encoding="utf-8",
    )
    status.append({"model": name, "seed": seed, "status": "complete", "seconds": elapsed})
    print(f"[COMPLETE] {name} seed={seed} in {elapsed / 60:.1f} min -> {result_path}")

print("\nStatus:")
for item in status:
    print(f"  {item['model']:55s} seed={item['seed']} {item['status']:18s} {item['seconds'] / 60:8.1f} min")


In [ ]:
# 5. Gộp mọi seed thành bảng mean ± sample std (thang [0, 100]).
import pandas as pd
from IPython.display import display

BENCHMARK_ORDER = [
    "banking77", "tweet", "emotion",
    "mrpc", "scitail", "wic",
    "sick", "sts12", "stsb",
]
SUMMARY_ORDER = ["avg_iod", "avg_ood", "avg_retrieval", "avg_all"]

rows = []
missing = []
for name in MODELS:
    for seed in SEEDS:
        result_path = result_path_of(name, seed)
        if not result_path.is_file():
            missing.append((name, seed, str(result_path)))
            continue
        payload = json.loads(result_path.read_text(encoding="utf-8"))["test"]
        row = {"model": name, "seed": seed}
        for family in ("classification", "pair", "sts", "retrieval"):
            for path, values in payload[family].items():
                row[KnowledgeDistiller._benchmark_name(path, "test")] = score_from_payload(family, values)
        for key in SUMMARY_ORDER:
            value = payload["summary"].get(key)
            row[key] = np.nan if value is None else float(value)
        # Tính lại từ raw score để cả result cũ cũng loại retrieval khỏi avg_all.
        row["avg_all"] = float(np.mean([row[name] for name in BENCHMARK_ORDER if name in row]))
        rows.append(row)

for name, seed, path in missing:
    print(f"[MISSING] {name} seed={seed}: {path}")

by_seed = pd.DataFrame(rows)
assert not by_seed.empty, "Không có results.json nào để gộp."
metric_order = [name for name in BENCHMARK_ORDER + SUMMARY_ORDER if name in by_seed.columns]
by_seed = by_seed[["model", "seed", *metric_order]].sort_values(["model", "seed"])


def mean_pm_std(mean, std, separator=" ± ", digits=2):
    # Cùng quy ước với 00_main_results: một seed thì không có sample std để in.
    if pd.isna(std):
        return f"{mean:.{digits}f}"
    return f"{mean:.{digits}f}{separator}{std:.{digits}f}"


grouped = by_seed.groupby("model", sort=False)[metric_order]
means, stds, ns = grouped.mean(), grouped.std(ddof=1), grouped.count()

wide = {}
for metric in metric_order:
    wide[f"{metric}_mean"] = means[metric]
    wide[f"{metric}_std"] = stds[metric]
    wide[f"{metric}_n"] = ns[metric]
mean_std_numeric = pd.DataFrame(wide)

paper_metrics = [name for name in BENCHMARK_ORDER + ["avg_all"] if name in metric_order]
paper_display = pd.DataFrame(index=means.index)
paper_latex = pd.DataFrame(index=means.index)
for metric in paper_metrics:
    paper_display[metric] = [
        mean_pm_std(mean * 100, std * 100) for mean, std in zip(means[metric], stds[metric])
    ]
    paper_latex[metric] = [
        mean_pm_std(mean * 100, std * 100, " $\\pm$ ") for mean, std in zip(means[metric], stds[metric])
    ]

by_seed.to_csv(RUN_ROOT / "baseline_by_seed.csv", index=False)
mean_std_numeric.to_csv(RUN_ROOT / "baseline_mean_std.csv")
paper_display.to_csv(RUN_ROOT / "baseline_mean_std_paper.csv")
(RUN_ROOT / "baseline_mean_std.tex").write_text(paper_latex.to_latex(escape=False), encoding="utf-8")
pd.DataFrame(status).to_csv(RUN_ROOT / "run_status.csv", index=False)

spread = (100 * (grouped.max() - grouped.min())).max().max()
print("BASELINE TEST — MỖI SEED (raw [0, 1])")
display(by_seed.style.format(precision=4))
label = "MEAN ± SAMPLE STD" if len(SEEDS) > 1 else "MEAN (một seed: không có std)"
print(f"BASELINE TEST — {label} (thang [0, 100])")
display(paper_display)
print(f"Chênh lệch max-min lớn nhất trên mọi (model, metric): {spread:.4f} điểm")
print(f"Saved to: {RUN_ROOT}")
